In [144]:
import os
import pandas as pd
from collections import defaultdict
import random

In [ ]:
# Books와 Movies 두 도메인의 데이터 읽기
triple_books = pd.read_csv("../../data/kg_half_hybrid_semi100pct_books.csv", sep=",").drop(['title'], axis=1)

print(f"Books triples: {len(triple_books)}")
triple_books.tail(5)

Books triples: 40348


,head_id:token,relation_id:token,tail_id:token
40343,res:Logan's_Run,dbo:adapted_into,cat:films
40344,res:Logan's_Run,dbo:adapted_into,cat:comics
40345,res:Logan's_Run,dbo:influence,cat:Marvel_Comics_universe
40346,res:Logan's_Run,dbo:sequel,res:Logan's_Run_sequels
40347,res:Logan's_Run,dbo:spinOff,res:Logan's_Run_spin_offs


In [ ]:

triple_movies = pd.read_csv("../../data/kg_half_hybrid_semi100pct_movies.csv", sep=",").drop(['title'], axis=1)
print(f"Movies triples: {len(triple_movies)}")
triple_movies.tail(5)




Movies triples: 48724


,head_id:token,relation_id:token,tail_id:token
48719,res:White_Lightnin',dbo:releaseDate,res:2009
48720,res:White_Lightnin',dbo:country,res:UK
48721,res:White_Lightnin',dbo:language,res:English
48722,res:White_Lightnin',dbo:genre,res:Drama
48723,res:Elephant_(film),dbo:releaseYear,res:2003


In [221]:


# 두 데이터프레임 합치기
triple_combined = pd.concat([triple_books, triple_movies], ignore_index=True)
print(f"Combined triples: {len(triple_combined)}")
triple_combined.head()

Combined triples: 89072


,head_id:token,relation_id:token,tail_id:token
0,res:Lig_Sinn_i_gCathú,dct:subject,cat:Novels_set_in_the_1940s
1,res:Lig_Sinn_i_gCathú,dct:subject,cat:1976_novels
2,res:Lig_Sinn_i_gCathú,dct:subject,cat:20th-century_Irish_novels
3,res:Lig_Sinn_i_gCathú,dct:subject,cat:Irish_historical_novels
4,res:Lig_Sinn_i_gCathú,dct:subject,cat:Irish-language_literature


In [222]:
# Books와 Movies 합쳐진 데이터에서 entities와 relations 수집
entities = set()
relations = set()
triples = []

# Books 도메인의 relation 추적 (common/specific 구분용)
books_relations = set()
movies_relations = set()

# Books 데이터 처리
for row in triple_books.itertuples(index=False):
    h, r, t = row[0], row[1], row[2]
    entities.add(h)
    entities.add(t)
    relations.add(r)
    books_relations.add(r)
    triples.append((h, r, t, 'books'))

    print(h, r, t)

  # 도메인 정보 추가

# Movies 데이터 처리
for row in triple_movies.itertuples(index=False):
    h, r, t = row[0], row[1], row[2]
    entities.add(h)
    entities.add(t)
    relations.add(r)
    movies_relations.add(r)
    triples.append((h, r, t, 'movies'))  # 도메인 정보 추가

# Common relations (두 도메인에 모두 나타나는 relation)
common_relations = books_relations & movies_relations
# Specific relations (한 도메인에만 있는 relation)
specific_relations = (books_relations | movies_relations) - common_relations

print(f"Total entities: {len(entities)}")
print(f"Total relations: {len(relations)}")
print(f"Books relations: {len(books_relations)}")
print(f"Movies relations: {len(movies_relations)}")
print(f"Common relations: {len(common_relations)}")
print(f"Specific relations: {len(specific_relations)}")

ent2id = {e: i for i, e in enumerate(sorted(entities))}
rel2id = {r: i for i, r in enumerate(sorted(relations))}


res:Lig_Sinn_i_gCathú dct:subject cat:Novels_set_in_the_1940s
res:Lig_Sinn_i_gCathú dct:subject cat:1976_novels
res:Lig_Sinn_i_gCathú dct:subject cat:20th-century_Irish_novels
res:Lig_Sinn_i_gCathú dct:subject cat:Irish_historical_novels
res:Lig_Sinn_i_gCathú dct:subject cat:Irish-language_literature
res:Lig_Sinn_i_gCathú dbo:author res:Breandán_Ó_hEithir
res:Lig_Sinn_i_gCathú dbo:literaryGenre res:Novel
res:Interzone_(magazine) dct:subject cat:Magazines_published_in_London
res:Interzone_(magazine) dct:subject cat:Bi-monthly_magazines_published_in_the_United_Kingdom
res:Interzone_(magazine) dct:subject cat:1982_establishments_in_the_United_Kingdom
res:Interzone_(magazine) dct:subject cat:Science_fiction_magazines_published_in_the_United_Kingdom
res:Interzone_(magazine) dct:subject cat:Magazines_established_in_1982
res:Interzone_(magazine) dct:subject cat:Mass_media_in_Leeds
res:Interzone_(magazine) dct:subject cat:Science_fiction_magazines_established_in_the_1980s
res:Interzone_(magazi

In [223]:
# random 샘플링 (10%만 사용)
random.seed(42) #재현성
random.shuffle(triples)
#num_train = int(len(triples) * 0.01)
train_triples = triples#[:num_train] # 앞 num_train 만큼 사용 (제거=뒷부분 무시)

print(f"Total triples: {len(triples)}")
print(f"Train triples (1%): {len(train_triples)}")


Total triples: 89072
Train triples (1%): 89072


In [224]:
# id 맵핑 (train_triples만으로, cold-start 시뮬)
train_entities = set()
train_relations = set()

for i in train_triples:
    h, r, t = i[0], i[1], i[2]
    train_entities.add(h)
    train_entities.add(t)
    train_relations.add(r)

# train_triples에 나타나는 entities와 relations만으로 ID 재매핑
ent2id = {e: i for i, e in enumerate(sorted(train_entities))}
rel2id = {r: i for i, r in enumerate(sorted(train_relations))}

print(f"Train entities: {len(ent2id)}")
print(f"Train relations: {len(rel2id)}")


Train entities: 59012
Train relations: 1373


In [142]:
# 파일 저장 (openKE 형식 :  h, r, t)
# entity2id.txt 저장
with open("entity2id.txt", "w") as f:
    f.write(f'{len(ent2id)}\n')
    for ent, eid in sorted(ent2id.items(), key=lambda x: x[1]):
        f.write(f'{ent}\t{eid}\n')

# relation2id.txt 저장
with open("relation2id.txt", "w") as f:
    f.write(f'{len(rel2id)}\n')
    for rel, rid in sorted(rel2id.items(), key=lambda x: x[1]):
        f.write(f'{rel}\t{rid}\n')

# train2id.txt 저장 (openKE 형식: h r t)
with open("train2id.txt", "w") as f:
    f.write(f'{len(train_triples)}\n')
    for triple in train_triples:
        h, r, t = triple[0], triple[1], triple[2]
        f.write(f'{ent2id[h]}\t{rel2id[r]}\t{ent2id[t]}\n')

# common_relations와 specific_relations 목록 저장 (학습 후 사용)
# train_triples에 나타나는 relation만 필터링
train_common_relations = common_relations & train_relations
train_specific_relations = specific_relations & train_relations

with open("common_relations.txt", "w") as f:
    f.write(f'{len(train_common_relations)}\n')
    for rel in sorted(train_common_relations):
        f.write(f'{rel}\n')

with open("specific_relations.txt", "w") as f:
    f.write(f'{len(train_specific_relations)}\n')
    for rel in sorted(train_specific_relations):
        f.write(f'{rel}\n')

print("Files saved:")
print("  - entity2id.txt, relation2id.txt, train2id.txt")
print(f"  - common_relations.txt ({len(train_common_relations)} relations)")
print(f"  - specific_relations.txt ({len(train_specific_relations)} relations)")


Files saved:
  - entity2id.txt, relation2id.txt, train2id.txt
  - common_relations.txt (168 relations)
  - specific_relations.txt (949 relations)


In [143]:
print(f'Original triples: {len(triples)}, Train triples: {len(train_triples)}')

Original triples: 97368, Train triples: 97368
